In [ ]:
import requests
import pandas as pd

api_url = "https://ows.goszakup.gov.kz/v3/refs/ref_amendm_agreem_justif"
token = "d5c3d78fc111d88a0a37b4ab8f83cbd5"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}
file = "amendment_agreement_justification_data.csv"

def retrieve_amendment_agreement_justification_data():
    """Получает данные справочника Основания создания дополнительного соглашения через API и сохраняет в CSV."""
    justification_list = []
    search_after = None

    while True:
        try:
            params = {"limit": 500}
            if search_after:
                params["page"] = "next"
                params["search_after"] = search_after

            response = requests.get(api_url, headers=headers, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            items = data.get("items", [])
            if not items:
                print("Все данные загружены.")
                break

            for item in items:
                justification_list.append({
                    "id": item.get("id"),
                    "name_kz": item.get("name_kz"),
                    "name_ru": item.get("name_ru"),
                    "cname_kz": item.get("cname_kz"),
                    "cname_ru": item.get("cname_ru")
                })

            print(f"Текущая загрузка: {len(justification_list)} записей")
            search_after = items[-1].get("id")
            if not data.get("next_page"):
                break

        except requests.exceptions.RequestException as e:
            print(f"Ошибка: {e}")
            break

    export_to_csv(justification_list)

def export_to_csv(dataset):
    """Сохраняет данные в CSV."""
    if not dataset:
        print("Нет данных для сохранения.")
        return
    df = pd.DataFrame(dataset)
    df.to_csv(file, index=False, encoding="utf-8-sig", sep="|", quotechar="'", escapechar="\\")
    print(f"Данные успешно сохранены в файл: {file} ({len(dataset)} записей)")

retrieve_amendment_agreement_justification_data()